In [1]:
import numpy as np
import networkx as nx
import pickle
import os
import json

from utils import *

%matplotlib inline
%load_ext autoreload
%autoreload 2

setup_environment()
set_paper_style()

In [ ]:
import json
import networkx as nx
import wikipedia
import re
import time

# ==============================================================================
# 設定エリア
# ==============================================================================
# 入力する知識グラフのJSONファイル
INPUT_JSON_PATH = "knowledge_graph_enhanced.json"
# Wikipedia情報を追加した、新しい知識グラフの出力ファイル
OUTPUT_JSON_WITH_WIKI_PATH = "knowledge_graph_with_wiki.json"
# ==============================================================================

# Wikipedia APIの言語を日本語に設定
wikipedia.set_lang("ja")


def get_wikipedia_summary(city_name):
    """
    市町村名を基にWikipediaの要約を取得し、整形する関数。
    """
    import wikipedia
    try:
        # まず市町村名でページを検索
        page = wikipedia.page(city_name, auto_suggest=False)
        summary = page.summary
        # 参照リンク[1]などを削除
        summary = re.sub(r"\[\d+\]", "", summary)
        # 改行をスペースに置換
        summary = summary.replace("\n", " ")
        return summary
    except wikipedia.exceptions.PageError:
        print(f"  - ⚠️ ページが見つかりません: {city_name}")
        return f"{city_name}のWikipediaページは見つかりませんでした。"
    except wikipedia.exceptions.DisambiguationError as e:
        # ページの曖昧さがある場合（例：「府中市」が東京と広島にある）
        print(f"  - ⚠️ ページの曖昧性あり: {city_name} -> {e.options[0]} を採用")
        return get_wikipedia_summary(e.options[0])
    except Exception as e:
        print(f"  - ❌ 不明なエラー: {city_name} ({e})")
        return f"{city_name}の要約取得中にエラーが発生しました。"


# --- 1. 既存の知識グラフを読み込む ---
print(f"🔄 '{INPUT_JSON_PATH}' を読み込んでいます...")
with open(INPUT_JSON_PATH, "r", encoding="utf-8") as f:
    graph_data = json.load(f)
# "edges"キーを使用するように指定
G = nx.json_graph.node_link_graph(graph_data, link="edges")
print("✅ グラフの読み込みが完了しました。")


# --- 2. 各市区町村ノードにWikipediaの要約テキストを追加 ---
print("🔄 各市区町村のWikipedia要約を取得し、ノードに追加しています...")
count = 0
for node_id, attrs in G.nodes(data=True):
    if attrs.get("type") == "市区町村":
        city_label = attrs.get("label")
        print(f"  -> {city_label} の情報を取得中...")

        summary = get_wikipedia_summary(city_label)
        G.nodes[node_id]["text"] = summary

        # APIへの連続アクセスを避けるための短い待機
        time.sleep(0.1)
        count += 1

print(f"✅ {count}件の市区町村ノードに要約テキストを追加しました。")

# --- 3. 新しいグラフをJSONファイルとして保存 ---
print(f"🔄 新しいグラフを '{OUTPUT_JSON_WITH_WIKI_PATH}' に保存しています...")
graph_json = nx.json_graph.node_link_data(G)
with open(OUTPUT_JSON_WITH_WIKI_PATH, "w", encoding="utf-8") as f:
    json.dump(graph_json, f, ensure_ascii=False, indent=2)
print("🎉 保存が完了しました。")

# --- 結果の確認 ---
print("\n--- テキスト追加後のノード例（札幌市） ---")
sapporo_node = G.nodes["R01100"]
print(f"ID: {sapporo_node.get('id')}")
print(f"Label: {sapporo_node.get('label')}")
print("--- 取得されたWikipedia要約 ---")
print(sapporo_node.get("text"))
print("---------------------------------")

🔄 'knowledge_graph_enhanced.json' を読み込んでいます...
✅ グラフの読み込みが完了しました。
🔄 各市区町村のWikipedia要約を取得し、ノードに追加しています...
  -> 札幌市 の情報を取得中...
  -> 函館市 の情報を取得中...
  -> 小樽市 の情報を取得中...
  -> 旭川市 の情報を取得中...
  -> 室蘭市 の情報を取得中...
  -> 釧路市 の情報を取得中...
  -> 帯広市 の情報を取得中...
  -> 北見市 の情報を取得中...
  -> 夕張市 の情報を取得中...
  -> 岩見沢市 の情報を取得中...
  -> 網走市 の情報を取得中...
  -> 留萌市 の情報を取得中...
  -> 苫小牧市 の情報を取得中...
  -> 稚内市 の情報を取得中...
  -> 美唄市 の情報を取得中...
  -> 芦別市 の情報を取得中...
  -> 江別市 の情報を取得中...
  -> 赤平市 の情報を取得中...
  -> 紋別市 の情報を取得中...
  -> 士別市 の情報を取得中...
  -> 名寄市 の情報を取得中...
  -> 三笠市 の情報を取得中...
  -> 根室市 の情報を取得中...
  -> 千歳市 の情報を取得中...
  -> 滝川市 の情報を取得中...
  -> 砂川市 の情報を取得中...
  -> 歌志内市 の情報を取得中...
  -> 深川市 の情報を取得中...
  -> 富良野市 の情報を取得中...
  -> 登別市 の情報を取得中...
  -> 恵庭市 の情報を取得中...
  -> 伊達市 の情報を取得中...
  - ⚠️ ページの曖昧性あり: 伊達市 -> 伊達市 (北海道) を採用
  -> 北広島市 の情報を取得中...
  -> 石狩市 の情報を取得中...
  -> 北斗市 の情報を取得中...
  -> 当別町 の情報を取得中...
  -> 新篠津村 の情報を取得中...
  -> 松前町 の情報を取得中...
  - ⚠️ ページの曖昧性あり: 松前町 -> 松前町 (北海道) を採用
  -> 福島町 の情報を取得中...
  -> 知内町 の情報を取得中...
  -> 木古内町 の